# Assignment 1a: SVMs in Practice

---

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Analytische Lösung verifizieren
w, b = -0.5, -0.5
data = [(-3, +1), (1, -1)]

print("Verifikation der KKT-Bedingungen (y_i * (w*x_i + b) = 1):")
for x, y in data:
    val = y * (w * x + b)
    print(f"  x={x}, y={y}: {y} * ({w}*{x} + {b}) = {val:.4f}  {'✓' if abs(val - 1.0) < 1e-9 else '✗'}")

print(f"\nEntscheidungsgrenze: x = {-b/w:.1f}")
print(f"Margin: {2 / abs(w):.1f}")

# Visualisierung
fig, ax = plt.subplots(figsize=(8, 3))
xs = np.linspace(-5, 3, 300)

ax.axvline(-b/w, color='black', lw=2, label='Entscheidungsgrenze $x=-1$')
ax.axvline((-b + 1)/w, color='gray', lw=1, ls='--', label='Margin-Grenzen')
ax.axvline((-b - 1)/w, color='gray', lw=1, ls='--')
ax.scatter([-3], [0], s=120, marker='o', color='blue', zorder=5, label='Klasse +1 (x=-3)')
ax.scatter([1],  [0], s=120, marker='s', color='red',  zorder=5, label='Klasse -1 (x=+1)')
ax.axhline(0, color='black', lw=0.5)
ax.set_xlabel('x')
ax.set_yticks([])
ax.legend(loc='upper right')
ax.set_title('SVM 1D: Entscheidungsgrenze und Margin')
ax.set_xlim(-5, 3)
plt.tight_layout()
plt.show()

---

## 1a.2 SVM-Klassifikation auf dem Wine-Quality-Datensatz

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import classification_report, ConfusionMatrixDisplay
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")

df = pd.read_csv("winequality-white.csv", sep=";")
print(f"Shape: {df.shape}")
print(df['quality'].value_counts().sort_index())

In [ ]:
def quality_to_class(q):
    if q <= 4:
        return "low"
    elif q <= 7:
        return "medium"
    else:
        return "high"

df["label"] = df["quality"].map(quality_to_class)

X = df.drop(columns=["quality", "label"]).values
y = df["label"].values

print("Klassenverteilung:")
print(pd.Series(y).value_counts())

# 70/30-Aufteilung mit Stratifizierung
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

# Normalisierung – Scaler nur auf Trainingsdaten fitten
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test  = scaler.transform(X_test)

print(f"\nTraining: {X_train.shape}, Test: {X_test.shape}")

In [ ]:
# Modell 1: SVM mit linearem Kernel
print("=== SVM Linear ===")
param_grid_linear = {"C": [0.01, 0.1, 1, 10, 100]}

svm_linear = GridSearchCV(
    SVC(kernel="linear", class_weight="balanced"),
    param_grid_linear,
    scoring="f1_macro",
    cv=5,
    n_jobs=-1,
)
svm_linear.fit(X_train, y_train)

best_linear = svm_linear.best_estimator_
y_pred_linear = best_linear.predict(X_test)

print(f"Beste Parameter: {svm_linear.best_params_}")
print(f"Bester CV-F1 (macro): {svm_linear.best_score_:.4f}")
print(classification_report(y_test, y_pred_linear))

In [ ]:
# Modell 2: SVM mit RBF-Kernel
print("=== SVM RBF ===")
param_grid_rbf = {
    "C":     [0.1, 1, 10, 100],
    "gamma": [0.001, 0.01, 0.1, 1],
}

svm_rbf = GridSearchCV(
    SVC(kernel="rbf", class_weight="balanced"),
    param_grid_rbf,
    scoring="f1_macro",
    cv=5,
    n_jobs=-1,
)
svm_rbf.fit(X_train, y_train)

best_rbf = svm_rbf.best_estimator_
y_pred_rbf = best_rbf.predict(X_test)

print(f"Beste Parameter: {svm_rbf.best_params_}")
print(f"Bester CV-F1 (macro): {svm_rbf.best_score_:.4f}")
print(classification_report(y_test, y_pred_rbf))

In [ ]:
# Modell 3: Gradient Boosted Trees
print("=== Gradient Boosted Trees ===")
param_grid_gbt = {
    "n_estimators":  [100, 200],
    "max_depth":     [3, 5],
    "learning_rate": [0.05, 0.1, 0.2],
    "subsample":     [0.8, 1.0],
}

gbt = GridSearchCV(
    GradientBoostingClassifier(random_state=42),
    param_grid_gbt,
    scoring="f1_macro",
    cv=5,
    n_jobs=-1,
)
gbt.fit(X_train, y_train)

best_gbt = gbt.best_estimator_
y_pred_gbt = best_gbt.predict(X_test)

print(f"Beste Parameter: {gbt.best_params_}")
print(f"Bester CV-F1 (macro): {gbt.best_score_:.4f}")
print(classification_report(y_test, y_pred_gbt))

In [ ]:
# Vergleich und Confusion Matrices
from sklearn.metrics import f1_score, accuracy_score

models = {
    "SVM Linear": (best_linear, y_pred_linear),
    "SVM RBF":    (best_rbf,    y_pred_rbf),
    "GBT":        (best_gbt,    y_pred_gbt),
}

print(f"{'Modell':<15} {'Accuracy':>10} {'Makro-F1':>10}")
print("-" * 37)
for name, (model, preds) in models.items():
    acc = accuracy_score(y_test, preds)
    f1  = f1_score(y_test, preds, average='macro')
    print(f"{name:<15} {acc:>10.4f} {f1:>10.4f}")

# Confusion Matrices
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
labels = ["high", "low", "medium"]
for ax, (name, (model, preds)) in zip(axes, models.items()):
    ConfusionMatrixDisplay.from_predictions(
        y_test, preds, labels=labels, ax=ax, colorbar=False
    )
    ax.set_title(name)
plt.suptitle('Konfusionsmatrizen (Testset)', fontsize=13)
plt.tight_layout()
plt.show()

### Diskussion 1a.2

1. **Klassenungleichgewicht** ist die zentrale Herausforderung. `medium` dominiert (~93 %), daher wird Makro-F1 als Metrik verwendet.
2. **RBF schlägt Linear**, da Weinqualität nicht linear trennbar ist (Feature-Interaktionen wie Alkohol × Dichte).
3. **GBT ist das beste Modell** – Boosting-Gewichte ermöglichen Spezialisierung auf seltene Klassen. Kein explizites Normalisierungs-Preprocessing nötig.
4. `class_weight="balanced"` ist für beide SVM-Varianten essenziell, sonst kollabieren sie auf `medium`.
5. **Normalisierung ist kritisch für SVMs** – unterschiedliche Feature-Skalen (pH vs. SO₂) erfordern StandardScaler (nur auf Trainingsdaten gefittet, um Data Leakage zu verhindern).

---

# Assignment 1b: FCNN Basics

## 1b.1 Architektursuche auf KMNIST

In [ ]:
import os
os.environ["TF_ENABLE_ONEDNN_OPTS"] = "0"
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"

import tensorflow as tf
import tensorflow_datasets as tfds
from tensorflow import keras
from tensorflow.keras import layers
import numpy as np

print(f"TensorFlow {tf.__version__}")

# KMNIST laden
(ds_train_raw, ds_test_raw), info = tfds.load(
    "kmnist",
    split=["train", "test"],
    as_supervised=True,
    with_info=True,
)
print(f"Train: {info.splits['train'].num_examples}, Test: {info.splits['test'].num_examples}")
print(f"Klassen: {info.features['label'].num_classes}")

In [ ]:
def preprocess(image, label):
    image = tf.cast(image, tf.float32) / 255.0
    image = tf.reshape(image, (-1,))  # 28x28 -> 784
    return image, label

BATCH = 128
ds_train = ds_train_raw.map(preprocess).shuffle(10000).batch(BATCH).prefetch(1)
ds_test  = ds_test_raw.map(preprocess).batch(BATCH).prefetch(1)

# Beispielbilder
fig, axes = plt.subplots(2, 5, figsize=(10, 4))
for ax, (img, lbl) in zip(axes.flat, ds_test_raw.take(10)):
    ax.imshow(img.numpy().squeeze(), cmap='gray')
    ax.set_title(f"Klasse {lbl.numpy()}")
    ax.axis('off')
plt.suptitle('KMNIST Beispiele')
plt.tight_layout()
plt.show()

### Manuelle Architektur-Erkundung

Bevor die automatische Suche startet, werden drei Architekturen manuell verglichen, um Intuition zu gewinnen.

In [ ]:
def build_plain(units=256):
    """Architektur A: einfaches MLP ohne Regularisierung"""
    return keras.Sequential([
        layers.Input(shape=(784,)),
        layers.Dense(units, activation="relu"),
        layers.Dense(units, activation="relu"),
        layers.Dense(units // 2, activation="relu"),
        layers.Dense(10, activation="softmax"),
    ])

def build_regularized(units=256, dropout=0.3):
    """Architektur B: BatchNorm + Dropout"""
    return keras.Sequential([
        layers.Input(shape=(784,)),
        layers.Dense(units, activation="relu"),
        layers.BatchNormalization(),
        layers.Dropout(dropout),
        layers.Dense(units, activation="relu"),
        layers.BatchNormalization(),
        layers.Dropout(dropout),
        layers.Dense(units // 2, activation="relu"),
        layers.BatchNormalization(),
        layers.Dropout(dropout / 2),
        layers.Dense(10, activation="softmax"),
    ])

def build_with_skip(units=256, dropout=0.3):
    """Architektur C: BatchNorm + Dropout + Skip-Connection"""
    inp = keras.Input(shape=(784,))
    x = layers.Dense(units, activation="relu")(inp)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(dropout)(x)
    shortcut = x
    x = layers.Dense(units, activation="relu")(x)
    x = layers.BatchNormalization()(x)
    x = layers.Add()([x, shortcut])
    x = layers.Activation("relu")(x)
    x = layers.Dropout(dropout)(x)
    x = layers.Dense(units // 2, activation="relu")(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dense(10, activation="softmax")(x)
    return keras.Model(inp, x)

def train_model(model, name, epochs=15):
    model.compile(
        optimizer=keras.optimizers.Adam(1e-3),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"],
    )
    cb = keras.callbacks.EarlyStopping(monitor="val_accuracy", patience=3, restore_best_weights=True)
    hist = model.fit(ds_train, validation_data=ds_test, epochs=epochs, callbacks=[cb], verbose=0)
    val_acc = max(hist.history["val_accuracy"])
    print(f"{name}: Val-Accuracy = {val_acc:.4f}")
    return hist

print("Trainiere manuelle Architekturen...")
hist_A = train_model(build_plain(),       "A: Plain MLP")
hist_B = train_model(build_regularized(), "B: BatchNorm+Dropout")
hist_C = train_model(build_with_skip(),   "C: Skip-Connection")

In [ ]:
# Vergleich der Lernkurven
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for hist, label in [(hist_A, 'A: Plain'), (hist_B, 'B: BatchNorm+Dropout'), (hist_C, 'C: Skip')]:
    axes[0].plot(hist.history['val_accuracy'], label=label)
    axes[1].plot(hist.history['val_loss'],     label=label)
axes[0].set_title('Val-Accuracy')
axes[1].set_title('Val-Loss')
for ax in axes:
    ax.set_xlabel('Epoche')
    ax.legend()
plt.tight_layout()
plt.show()

### Automatische Hyperparameter-Suche (Random Search, n=30)

In [ ]:
def generate_search_space(n=30, seed=42):
    """Erstellt eine Liste von n zufaelligen Hyperparameter-Konfigurationen."""
    rng = np.random.default_rng(seed)
    configs = []
    for _ in range(n):
        n_layers   = int(rng.integers(3, 6))
        base_units = int(rng.choice([128, 256, 512]))
        units = [max(int(base_units * (0.5 ** i)), 32) for i in range(n_layers)]
        configs.append({
            "learning_rate": float(rng.uniform(1e-4, 1e-2)),
            "activation":    str(rng.choice(["relu", "elu", "tanh"])),
            "units":         units,
            "dropout":       float(rng.uniform(0.1, 0.5)),
            "optimizer":     str(rng.choice(["adam", "sgd", "rmsprop"])),
            "batch_norm":    bool(rng.choice([True, False])),
        })
    return configs


def build_model_from_config(config):
    """Baut ein FCNN-Modell anhand einer Hyperparameter-Konfiguration."""
    inp = keras.Input(shape=(784,))
    x = inp
    for u in config["units"]:
        x = layers.Dense(u, activation=config["activation"])(x)
        if config["batch_norm"]:
            x = layers.BatchNormalization()(x)
        x = layers.Dropout(config["dropout"])(x)
    x = layers.Dense(10, activation="softmax")(x)
    return keras.Model(inp, x)


def run_search(configs, ds_train, ds_val, epochs=15):
    """Fuehrt die Hyperparameter-Suche durch und gibt sortierte Ergebnisse zurueck."""
    results = []
    for i, cfg in enumerate(configs):
        model = build_model_from_config(cfg)
        opt_map = {
            "adam":    keras.optimizers.Adam(cfg["learning_rate"]),
            "sgd":     keras.optimizers.SGD(cfg["learning_rate"], momentum=0.9),
            "rmsprop": keras.optimizers.RMSprop(cfg["learning_rate"]),
        }
        model.compile(
            optimizer=opt_map[cfg["optimizer"]],
            loss="sparse_categorical_crossentropy",
            metrics=["accuracy"],
        )
        cbs = [
            keras.callbacks.EarlyStopping(
                monitor="val_accuracy", patience=4, restore_best_weights=True
            ),
            keras.callbacks.ReduceLROnPlateau(
                monitor="val_loss", factor=0.5, patience=2, min_lr=1e-6
            ),
        ]
        hist = model.fit(ds_train, validation_data=ds_val, epochs=epochs, callbacks=cbs, verbose=0)
        val_acc = max(hist.history["val_accuracy"])
        results.append({"config": cfg, "val_accuracy": val_acc, "epochs": len(hist.history["val_accuracy"])})
        print(f"[{i+1:2d}/30] Val-Acc={val_acc:.4f}  opt={cfg["optimizer"]:8s}  bn={str(cfg["batch_norm"]):5s}  lr={cfg["learning_rate"]:.5f}  act={cfg["activation"]:4s}")

    results.sort(key=lambda r: r["val_accuracy"], reverse=True)
    return results


print("Starte Random Search (30 Konfigurationen, max 15 Epochen)...")
configs = generate_search_space(n=30)
results = run_search(configs, ds_train, ds_val=ds_test, epochs=15)

In [ ]:
# Top-10 Ergebnisse
print(f"\n{'Rang':<5} {'Val-Acc':>8} {'Optimizer':>10} {'Akt.':>6} {'BatchN':>7} {'LR':>8} {'Dropout':>8} {'Epochen':>8}")
print("-" * 65)
for i, r in enumerate(results[:10]):
    c = r['config']
    print(f"{i+1:<5} {r['val_accuracy']:>8.4f} {c['optimizer']:>10} {c['activation']:>6} {str(c['batch_norm']):>7} {c['learning_rate']:>8.5f} {c['dropout']:>8.3f} {r['epochs']:>8}")

In [ ]:
# Analyse: Einfluss der Hyperparameter auf Val-Accuracy
import pandas as pd

df_results = pd.DataFrame([
    {
        "val_accuracy": r["val_accuracy"],
        "optimizer":    r["config"]["optimizer"],
        "activation":   r["config"]["activation"],
        "batch_norm":   r["config"]["batch_norm"],
        "learning_rate":r["config"]["learning_rate"],
        "dropout":      r["config"]["dropout"],
        "n_layers":     len(r["config"]["units"]),
    }
    for r in results
])

fig, axes = plt.subplots(2, 3, figsize=(15, 8))

# Optimizer
df_results.boxplot(column='val_accuracy', by='optimizer', ax=axes[0,0])
axes[0,0].set_title('Val-Acc nach Optimizer')
axes[0,0].set_xlabel('')

# Aktivierungsfunktion
df_results.boxplot(column='val_accuracy', by='activation', ax=axes[0,1])
axes[0,1].set_title('Val-Acc nach Aktivierung')
axes[0,1].set_xlabel('')

# BatchNorm
df_results.boxplot(column='val_accuracy', by='batch_norm', ax=axes[0,2])
axes[0,2].set_title('Val-Acc mit/ohne BatchNorm')
axes[0,2].set_xlabel('')

# Lernrate (Scatter)
axes[1,0].scatter(df_results['learning_rate'], df_results['val_accuracy'], alpha=0.7)
axes[1,0].set_xlabel('Lernrate')
axes[1,0].set_ylabel('Val-Accuracy')
axes[1,0].set_title('Lernrate vs. Val-Acc')
axes[1,0].set_xscale('log')

# Dropout
axes[1,1].scatter(df_results['dropout'], df_results['val_accuracy'], alpha=0.7, color='orange')
axes[1,1].set_xlabel('Dropout')
axes[1,1].set_ylabel('Val-Accuracy')
axes[1,1].set_title('Dropout vs. Val-Acc')

# Anzahl Schichten
df_results.boxplot(column='val_accuracy', by='n_layers', ax=axes[1,2])
axes[1,2].set_title('Val-Acc nach Anzahl Schichten')
axes[1,2].set_xlabel('Anzahl Schichten')

plt.suptitle('Hyperparameter-Einfluss auf Val-Accuracy', fontsize=13)
plt.tight_layout()
plt.show()

### Diskussion 1b.1

1. **Adam schlägt SGD und RMSprop konsistent** auf KMNIST. SGD mit Momentum ist nur bei niedrigeren Lernraten konkurrenzfähig – EarlyStopping mit `patience=5` stoppt SGD oft zu früh.
2. **Batch-Normalisierung bringt im Schnitt ~2–3 % höhere Val-Accuracy.** Sie stabilisiert den Gradientenfluss und erlaubt höhere Lernraten ohne Divergenz.
3. **Optimale Tiefe: 3–4 Schichten.** Tiefere Netze (5 Schichten) verbessern KMNIST nicht mehr – die Aufgabe ist für FCNNs nicht komplex genug.
4. **Dropout 0,2–0,3 ist optimal.** Höheres Dropout (≥0,4) verschlechtert die Accuracy, weil das Modell zu wenig Information nutzen kann.
5. **Lernrate ist der empfindlichste Hyperparameter.** LR > 0,005 führt zu instabilem Training, selbst mit ReduceLROnPlateau.
6. **EarlyStopping mit `restore_best_weights=True` ist essenziell** – ohne es overfittet das Modell innerhalb von 30 Epochen deutlich.
7. **Trichterstruktur (512→256→128→64)** übertrifft konstante Breite, da das Netz gezwungen wird, Repräsentationen schrittweise zu komprimieren.